<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "834"
iso_country = "Tanzania, United Republic of"
year = 2025
import_portfolio = "h6_ag6_by_sea_M*.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h6_ag6_by_sea_X*.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
gdp = 90.140*1e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Tanzania, United Republic of': 

Number of trade records present in dataset: 47787
Number of partner countries present in dataset: 187
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$10562.7m
Total weight within the dataset: 9742.7 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,834,United Rep. of Tanzania,646,Rwanda,Import,Sea,392210,"392210 | Plastics; baths, sho",3922,3922 | Sanitary ware; baths,39,39 | Plastics and article,3693,"Baths, wash-basins, lavatory pans and covers, ...","Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",19.304707,NaN,19.304707,kg,0.563,0.563,3.216164,0.0,0.000000,0.000000,['Container'],['Container'],0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3693
1,1,834,United Rep. of Tanzania,620,Portugal,Import,Sea,392210,"392210 | Plastics; baths, sho",3922,3922 | Sanitary ware; baths,39,39 | Plastics and article,3693,"Baths, wash-basins, lavatory pans and covers, ...","Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",3376.925796,NaN,3376.925796,kg,5.630,5.630,562.595838,10075.0,6260.312825,3522.025938,['Container'],['Container'],12.456371,12.416922,12.420162,12.420162,12.443398,12.443398,13.080415,24.279001,14.311303,13.36129,13.40954,13.752606,12.784402,23.250933,15.043647,13.867758,13.536394,14.212668,12.524957,25.81984,18.869476,16.464953,15.200402,14.847788,12.269533,27.352141,22.114418,20.530151,16.146317,15.960645,12.018836,27.985919,26.678078,27.14981,17.369873,16.785214,11.763252,27.461094,26.045243,27.840295,21.241546,17.104834,11.523075,24.213186,25.202445,26.30016,19.214749,17.311954,11.278214,21.985252,22.381623,25.741709,21.466873,17.445563,3693


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $923.9m (8.7%), ... 

                              cifvalue    vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                              
10 | Cereals               310553643.8  1.139630e+09      2.9     11.7
17 | Sugars and sugar con  183914775.2  3.695489e+08      1.7      3.8
15 | Animal, vegetable or  145769095.1  1.517616e+08      1.4      1.6
23 | Food industries, res   73553266.6  9.789131e+07      0.7      1.0
11 | Products of the mill   62234755.3  1.149088e+08      0.6      1.2
22 | Beverages, spirits a   56759101.5  4.020578e+07      0.5      0.4
21 | Miscellaneous edible   22350358.4  6.201325e+06      0.2      0.1
19 | Preparations of cere   13861764.6  5.904817e+06      0.1      0.1
12 | Oil seeds and oleagi   11791106.4  5.304541e+06      0.1      0.1
08 | Fruit and nuts, edib    6928402.7  5.825956e+06      0.1      0.1
20 | Preparations of vege    5833249.3  4.642081

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $585.4m (5.5%), ... 

                     cifvalue   vol_kg_vtw  pct_val  pct_vol
eora_desc                                                   
Food & Beverages  337833662.6  471765644.2      3.2      4.8
Agriculture       247460689.7  370734404.2      2.3      3.8
Fishing               75433.5      18857.7      0.0      0.0



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $498.3m (4.7%), ... 

                                                 cifvalue   vol_kg_vtw  \
gtap_desc                                                                
VOL - Vegetable oils and fats                 151832885.7  203723777.2   
OFD - Food products n.e.c.                    142227073.7  117802750.9   
SGR - Sugar                                   139015604.8  305885792.5   
GRO - Cereal grains n.e.c.                     22743126.9   94076229.2   
OCR - Crops n.e.c.                             17465825.7    3969561.4   
V_F - Vegetables, fruit, nuts                   8468660.2    8738361.8   
MIL - Dairy products                            6901815.5    2997357.1   
OMT - Meat products n.e.c.                      6552411.2    2815012.5   
OSD - Oil seeds                                 1265975.3    1630358.3   
PDR - Paddy rice                                 804888.4    1398091.1   
CMT - Bovine meat prods                        

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 8.7% 
GTAP 4.7% 
EORA 5.5%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_46428/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,834,United Rep. of Tanzania,646,Rwanda,Import,Sea,392210,"392210 | Plastics; baths, sho",3922,3922 | Sanitary ware; baths,39,39 | Plastics and article,3693,"Baths, wash-basins, lavatory pans and covers, ...","Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",19.304707,NaN,19.304707,kg,0.563,0.563,3.216164,0.0,0.000000,0.000000,['Container'],['Container'],0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3693,NaN
1,1,834,United Rep. of Tanzania,620,Portugal,Import,Sea,392210,"392210 | Plastics; baths, sho",3922,3922 | Sanitary ware; baths,39,39 | Plastics and article,3693,"Baths, wash-basins, lavatory pans and covers, ...","Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",3376.925796,NaN,3376.925796,kg,5.630,5.630,562.595838,10075.0,6260.312825,3522.025938,['Container'],['Container'],12.456371,12.416922,12.420162,12.420162,12.443398,12.443398,13.080415,24.279001,14.311303,13.36129,13.40954,13.752606,12.784402,23.250933,15.043647,13.867758,13.536394,14.212668,12.524957,25.81984,18.869476,16.464953,15.200402,14.847788,12.269533,27.352141,22.114418,20.530151,16.146317,15.960645,12.018836,27.985919,26.678078,27.14981,17.369873,16.785214,11.763252,27.461094,26.045243,27.840295,21.241546,17.104834,11.523075,24.213186,25.202445,26.30016,19.214749,17.311954,11.278214,21.985252,22.381623,25.741709,21.466873,17.445563,3693,NaN


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $648.7m of $10562.7m (6.1%)
Volume: 1660.1 of 9742.7 million kilograms (17.0%)
Value-to-Weight: 0.39 from 1.08 US$ per kg (+-64.0%)
Average distance to market (volume-weighted): 5538.8 from 4548.1 nm (+21.8%)

            food_group  N_com     cifvalue    vol_kg_vtw  pct_val  pct_vol
0  Unprocessed Cereals     59  309739823.4  1.138217e+09      2.9     11.7
1      Sugar and Honey    145  183919182.9  3.695502e+08      1.7      3.8
2    Processed Cereals    660   86778754.6  1.091854e+08      0.8      1.1
3        Fats and Oils    201   38790526.4  2.585102e+07      0.4      0.3
4           Vegetables    321   12534506.4  6.674826e+06      0.1      0.1
5               Fruits    183    6919451.4  5.821799e+06      0.1      0.1
6                Dairy    130    5719074.9  2.529032e+06      0.1      0.0
7        Meat and Prep     55    4322218.8  2.305091e+06      0.0      0.0



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,1.177263e+06,2.164799e+06,1.284865e+06,1.200495e+06,1.205012e+06,1.236424e+06,1.083474e+06,2.498701e+06,2.379895e+06,2.427492e+06,1.557752e+06,1.504081e+06,1.018127e+06,1.967049e+06,2.001998e+06,2.302762e+06,1.921397e+06,1.563857e+06
1,Fats and Oils,3.558395e+07,6.249379e+07,3.837190e+07,3.598906e+07,3.615276e+07,3.717751e+07,3.300299e+07,7.254348e+07,6.882021e+07,7.104419e+07,4.628790e+07,4.449725e+07,3.121930e+07,5.777650e+07,5.873839e+07,6.761693e+07,5.646564e+07,4.636435e+07
2,Fruits,3.735310e+06,6.911930e+06,4.083272e+06,3.813314e+06,3.827346e+06,3.925823e+06,3.433986e+06,7.971424e+06,7.595698e+06,7.736723e+06,4.954391e+06,4.786205e+06,3.223892e+06,6.266497e+06,6.379194e+06,7.337465e+06,6.118690e+06,4.975555e+06
3,Meat and Prep,5.834502e+06,1.082962e+07,6.383538e+06,5.959785e+06,5.981307e+06,6.134331e+06,5.360986e+06,1.248308e+07,1.189972e+07,1.211014e+07,7.747808e+06,7.487022e+06,5.030632e+06,9.806492e+06,9.983293e+06,1.148205e+07,9.575270e+06,7.781570e+06
4,Processed Cereals,2.284603e+08,4.214932e+08,2.495612e+08,2.331322e+08,2.339822e+08,2.400392e+08,2.101451e+08,4.864566e+08,4.636115e+08,4.722743e+08,3.030720e+08,2.923232e+08,1.973728e+08,3.826996e+08,3.895444e+08,4.479062e+08,3.736741e+08,3.037772e+08
5,Sugar and Honey,4.532187e+08,7.879147e+08,4.879178e+08,4.584473e+08,4.599914e+08,4.732842e+08,4.212271e+08,9.220596e+08,8.799406e+08,9.011413e+08,6.000649e+08,5.640492e+08,3.989057e+08,7.362035e+08,7.480728e+08,8.556660e+08,7.188628e+08,5.832215e+08
6,Unprocessed Cereals,2.636532e+09,4.877141e+09,2.882156e+09,2.691744e+09,2.701429e+09,2.771017e+09,2.424050e+09,5.626123e+09,5.363503e+09,5.460088e+09,3.500597e+09,3.377819e+09,2.275808e+09,4.423470e+09,4.502784e+09,5.177296e+09,4.319176e+09,3.509760e+09
7,Vegetables,7.359900e+06,1.332492e+07,7.996823e+06,7.483316e+06,7.514344e+06,7.715436e+06,6.791487e+06,1.542649e+07,1.465548e+07,1.501858e+07,9.682926e+06,9.333352e+06,6.396673e+06,1.218546e+07,1.240129e+07,1.427213e+07,1.189733e+07,9.716255e+06


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.17,0.02,0.00,0.00,0.01,0.25,0.23,0.24,0.08,0.07,0.17,0.17,0.22,0.16,0.10
1,Fats and Oils,0.69,0.07,0.01,0.01,0.04,1.02,0.92,0.98,0.34,0.30,0.68,0.71,0.94,0.65,0.39
2,Fruits,0.46,0.05,0.01,0.01,0.03,0.66,0.60,0.62,0.22,0.20,0.44,0.46,0.59,0.42,0.25
3,Meat and Prep,1.16,0.13,0.03,0.03,0.07,1.65,1.51,1.56,0.55,0.49,1.10,1.15,1.49,1.05,0.64
4,Processed Cereals,2.22,0.24,0.05,0.06,0.13,3.18,2.92,3.02,1.07,0.95,2.14,2.21,2.89,2.03,1.23
5,Sugar and Honey,1.82,0.19,0.03,0.04,0.11,2.72,2.49,2.61,0.97,0.78,1.83,1.90,2.48,1.74,1.00
6,Unprocessed Cereals,7.23,0.79,0.18,0.21,0.43,10.34,9.49,9.80,3.48,3.08,6.93,7.19,9.37,6.60,3.98
7,Vegetables,0.48,0.05,0.01,0.01,0.03,0.69,0.63,0.66,0.23,0.20,0.46,0.48,0.63,0.44,0.26
